In [1]:
!hpc-ticket
!hpc-mount wissdaten

# Graphstructures

## Standard epidataloader

In [2]:
from src.utils import get_data_env
from src.dataloading import EpiConfig, DataOrchestrator, ShallowDataLoaderManager, GraphDataLoaderManager
from src.models import PersistenceModel, NodeRFModel

disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        = '2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False

horizon_size    = 4
horizon_leadtime= 1
sequence_length = 1
lag_num         = 1

config = EpiConfig(
    disease             = 'influenza',
    data_env_dir        = get_data_env(),
    date_range          = (min_date, max_date),
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = nuts_level,
    log_transform       = ['incidence'],
    split_berlin        = split_berlin,
    include_population  = False,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'incidence',
    lag_column          = 'incidence',    
    )    
data_orchestrator = DataOrchestrator(config).build()

target: incidence
Index(['timestamp', 'node', 'incidence', 'timestamp_sin', 'timestamp_cos',
       'incidence_lag0', 'train', 'val', 'test', 'target_ahead1',
       'target_ahead2', 'target_ahead3', 'target_ahead4'],
      dtype='object')


In [ ]:
import os
from typing import Optional, Tuple, List, Union, Literal, Dict
import pandas as pd
from io import StringIO
from src.utils import get_data_env
from src.utils.textformatting import checkmark
from tqdm import tqdm


dir_commuting_raw   = os.path.join(get_data_env(),'raw/germany/mobility/commuter_data/auspendler/')
dir_commuting_pcd   = os.path.join(get_data_env(),'processed/germany/mobility/commuter_data/')
dir_harmfile        = os.path.join(get_data_env(),'processed/germany/geospatial/harmonization/german_nuts_harmonization.tsv')


class CommuterDataProcessor:
    """
    Processor of commuter-data
    
    Returns single pd.DataFrame with the number of commuters between two Kreisen.

    Examples:
    --------
    >>> pr = CommuterDataProcessor()
    >>> # currently only 2024 is available!
    >>> pr.process_data('2024')
    """
    def __init__(self):
        self.harmfile       = pd.read_csv(dir_harmfile, sep ="\t", dtype=str)
        self.rename_cols    = {'Regionalschlüssel'  : 'nuts3_work',
                               'Regionalschlüssel.1': 'nuts3_residence',
                               'Insgesamt'          : 'commuters'}
        self.columns        = list(self.rename_cols.values())
        self.data           = None

    def process_data(self, years: Union[List[str], str]):
        """ 
        Loop over all years (folder) to return a merged df.

        Parameters:
        ----------
        years: Union[List[str], str]:
            list of years over which to loop

        Returns:
        -------
        pd.DataFrame   
            df with columns: 'nuts3_work', 'nuts3_residence','commuters','year'
        """
        # make years iterable if it isn't
        if isinstance(years, str):
            years = [years]

        # looping over years
        for ii, yy in tqdm(enumerate(years), desc = 'compiling data from years', total=len(years)):
            # get concatenated df
            yearly_df = self._concatenate_yearly_data(yy)
            if ii == 0:
                all_data = yearly_df
            else:
                all_data = pd.concat([all_data, yearly_df], ignore_index=True)  # type: ignore => all_data will not be unbound

        self.data = all_data # type: ignore
        print(f'{checkmark} data processed for {years}')

    def save_data(self):
        """ 
        saves self.data
        """
        if self.data is not None:
            self.data.to_csv(os.path.join(dir_commuting_pcd,'commuting_data.csv'), index=False)
            print(f'{checkmark} data saved ')

    def _concatenate_yearly_data(self, year: str) -> pd.DataFrame:
        """ 
        clean and concatenate all datafiles for a year
        """
        rawfolder = os.path.join(dir_commuting_raw, year)
        all_data = []  # to accumulate all processed DataFrames

        for file in os.listdir(rawfolder):  # iterate files in the folder
            
            path        = os.path.join(rawfolder, file)
            trimmed_data= clean_csv_file(path)

            if trimmed_data is None:
                raise ImportError(f'No data found inside {path}')

            trimmed_csv = pd.read_csv(trimmed_data, sep=";", dtype={'Regionalschlüssel'  : 'str', 'Regionalschlüssel.1': 'str'}).rename(columns=self.rename_cols)
            trimmed_csv = trimmed_csv[self.columns]
            
            unique_nuts3 = list(self.harmfile['nuts3'].unique())
                
            filtered_csv = trimmed_csv[
                trimmed_csv['nuts3_work'].isin(unique_nuts3) & 
                trimmed_csv['nuts3_residence'].isin(unique_nuts3)
            ]

            all_data.append(filtered_csv)  # append the processed dataframe

        df          = pd.concat(all_data, ignore_index=True)
        df['year']  = year
        return df
    
def clean_csv_file(path: str) -> Optional[StringIO]:

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    start_index = next((i for i, line in enumerate(lines) if line.count(';') > 3), None)
        
    if start_index is not None:
        # Slice and remove trailing garbage lines
        trimmed_lines = lines[start_index:]
        valid_lines   = [line for line in trimmed_lines if line.count(';') > 3]

        cleaned_lines = [line.replace('.','') for line in valid_lines]

        # Create a temporary in-memory CSV
        return StringIO(''.join(cleaned_lines))

    else:
        return None

from pathlib import Path
from srcv2.utils.textformatting import warning_emoji, checkmark

class RawCommuterDataValidator:
    
    def __init__(self, raw_dir: Union[str,Path]):
        if isinstance(raw_dir, str):
            raw_dir = Path(raw_dir)    

        self.raw_root_dir       = raw_dir 
        self.expected_filenames = [
            "bw", "by", "be", "bb", "hb", "hh", "he", "ni", 
            "mv", "nw", "rp", "sl", "sn", "st", "sh", "th"
        ]
        self.expected_filenumbers = len(self.expected_filenames)


    def _find_subfolders(self) -> List[str]:
        """Find all subdirectories that appear to be year folders."""
        year_absolute_folders = sorted([f for f in self.raw_root_dir.iterdir() if f.is_dir()])
        year_folders          = [str(folder_path).split("/")[-1] for folder_path in year_absolute_folders]
        self.absolute_paths   = year_absolute_folders
        self.relative_paths   = year_folders
        return year_folders

    def _count_files_per_subdir(self):
        check          = True
        check_report   = ""
        for ii, folder  in enumerate(self.absolute_paths):
            number_files = len(list(folder.glob('*.csv')))
            
            if number_files != self.expected_filenumbers:
                check = False
                check_report += f'In {self.relative_paths[ii]} I found {number_files} files - expected: {self.expected_filenumbers}\n'

        if check:
            print(f'{checkmark} 16 files in each folder')
        else:
            print(f'{warning_emoji} number of files per folder:')
            print(check_report)
            
    def _check_filenames(self):
        pass


class CommuterDataLoader:
    """ 
    Simple dataloader object to return commuting_data

    Parameters:
    ----------
    years: Union[List[str], str]
        the years for which to select data
    """
    def __init__(self, years: Union[List[str],str]):
        if isinstance(years, str):
            years = [years]
        self.years = years

    def import_data(self) -> pd.DataFrame:
        """imports and returns the dataframe of selected years"""
        df = pd.read_csv(os.path.join(dir_commuting_pcd, 'commuting_data.csv'), dtype={'nuts3_work':'str','nuts3_residence':'str','year':'str'})
        df = df[df['year'].isin(self.years)]
        return df



In [4]:
years_int = range(2002,2025)
years_str = [str(yy) for yy in years_int]
xyz = CommuterDataProcessor()
xyz.process_data(years = years_str)

compiling data from years: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:04<00:00,  5.51it/s]

✓ data processed for ['2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


In [ ]:
from src.dataloading.graphconstruction import GraphOrchestrator

orch                = data_orchestrator
graphconstruction   = GraphOrchestrator(data_orchestrator=data_orchestrator)
graphconstruction.generate_graph('commuter', scaling_method='rowwise')
graphconstruction.rename_graph('commuter_selfmean_rowwise', 'commuter')
graphconstruction.generate_graph('boolean_neighbors')
graphconstruction.rename_graph('boolean_neighbors_selfmean', 'boolean_neighbors')

graphconstruction.preview_graph('commuter', node_idx = 223, subplots = True, title= "Preview commuter graph from Munich")

✓ graph generated: commuter_selfmean_rowwise
commuter_selfmean_rowwise has been replaced by commuter


In [ ]:
graphconstruction.preview_graph('boolean_neighbors', subplots = True, title= "Preview commuter graph from Munich")